# 05 - End-to-end pipeline demo: folder of images -> detector -> severity -> Llama -> log  (run on the Jetson, offline)

```
image (200x200 crop)
   |  ONNX Runtime (YOLOv8n)            -> boxes, classes, confidence
   |  severity.py                       -> Low / Medium / High per defect, image severity, zones
   |  hand-off JSON                     -> only the MAJOR defect goes to Llama
   |  llama_main (ExecuTorch, 4-bit)    -> explanation, actions, log_entry (or template fallback)
   v
runs/<run>/inspections.jsonl  +  annotated images  (+ optional POST to the tracker on localhost)
```
Nothing here needs the internet. The tracker URL is `localhost`.

**Before running:** `deploy/` (from notebook 00) and `llama_deploy/` + `bin/llama_main` (notebooks 01 and 02) must be in the repo. Optional: start the tracker in a terminal
(`cd tracker && python3 app.py --media ../runs`) and open <http://127.0.0.1:5000>.

In [ ]:
import sys, json, time, importlib
from pathlib import Path
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))

In [ ]:
%%writefile ../edge_inspect/detector.py
"""detector.py - YOLOv8n ONNX detector (ONNX Runtime; GPU via CUDA provider if present, else CPU).

Python 3.8+ compatible. Needs: numpy, opencv-python (cv2), onnxruntime(-gpu).
Per-stage timings (pre / inference / post) are returned for the benchmark.
"""
from __future__ import annotations

import time
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import cv2
except ImportError:  # pragma: no cover
    cv2 = None


def letterbox(img, size: int = 416, pad_value: int = 114):
    """Resize keeping aspect ratio onto a size x size canvas. Returns (canvas, ratio, (pad_x, pad_y))."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas[py:py + nh, px:px + nw] = resized
    return canvas, r, (px, py)


def nms_numpy(boxes: np.ndarray, scores: np.ndarray, iou_thr: float) -> List[int]:
    order = scores.argsort()[::-1]
    keep: List[int] = []
    while order.size:
        i = order[0]
        keep.append(int(i))
        if order.size == 1:
            break
        rest = order[1:]
        xx1 = np.maximum(boxes[i, 0], boxes[rest, 0])
        yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])
        xx2 = np.minimum(boxes[i, 2], boxes[rest, 2])
        yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])
        inter = np.maximum(0.0, xx2 - xx1) * np.maximum(0.0, yy2 - yy1)
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_r = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])
        iou = inter / (area_i + area_r - inter + 1e-9)
        order = rest[iou <= iou_thr]
    return keep


def postprocess(raw: np.ndarray, n_classes: int, conf: float, iou: float,
                ratio: float, pad: Tuple[int, int], orig_hw: Tuple[int, int]) -> List[dict]:
    """raw: YOLOv8 head output (1, 4+nc, N) or (1, N, 4+nc). Returns detections in ORIGINAL image pixels."""
    p = raw[0]
    if p.shape[0] < p.shape[1]:       # (4+nc, N) -> (N, 4+nc)
        p = p.T
    boxes_cxcywh, scores = p[:, :4], p[:, 4:4 + n_classes]
    cls = scores.argmax(1)
    conf_all = scores.max(1)
    m = conf_all >= conf
    if not m.any():
        return []
    b, c, s = boxes_cxcywh[m], cls[m], conf_all[m]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
    keep = nms_numpy(xyxy + c[:, None] * 4096.0, s, iou)      # class-aware NMS via coordinate offset
    h0, w0 = orig_hw
    out = []
    for i in keep:
        x1 = (xyxy[i, 0] - pad[0]) / ratio
        y1 = (xyxy[i, 1] - pad[1]) / ratio
        x2 = (xyxy[i, 2] - pad[0]) / ratio
        y2 = (xyxy[i, 3] - pad[1]) / ratio
        out.append({"class_id": int(c[i]), "confidence": float(s[i]),
                    "xyxy": [float(np.clip(x1, 0, w0)), float(np.clip(y1, 0, h0)),
                             float(np.clip(x2, 0, w0)), float(np.clip(y2, 0, h0))]})
    return out


class OnnxYoloDetector:
    def __init__(self, onnx_path: str, class_names: Sequence[str], imgsz: int = 416,
                 conf: float = 0.25, iou: float = 0.5, prefer_gpu: bool = True,
                 intra_threads: Optional[int] = None):
        import onnxruntime as ort
        so = ort.SessionOptions()
        if intra_threads:
            so.intra_op_num_threads = intra_threads
        avail = ort.get_available_providers()
        providers = [p for p in (["CUDAExecutionProvider"] if prefer_gpu else []) if p in avail] + ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(onnx_path, sess_options=so, providers=providers)
        self.providers = self.session.get_providers()
        self.input_name = self.session.get_inputs()[0].name
        self.class_names = list(class_names)
        self.imgsz, self.conf, self.iou = imgsz, conf, iou

    def infer(self, bgr: np.ndarray) -> Tuple[List[dict], Dict[str, float]]:
        t0 = time.perf_counter()
        canvas, ratio, pad = letterbox(bgr, self.imgsz)
        x = canvas[:, :, ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0
        x = np.ascontiguousarray(x)
        t1 = time.perf_counter()
        raw = self.session.run(None, {self.input_name: x})[0]
        t2 = time.perf_counter()
        dets = postprocess(raw, len(self.class_names), self.conf, self.iou, ratio, pad, bgr.shape[:2])
        for d in dets:
            d["class_name"] = self.class_names[d["class_id"]]
        t3 = time.perf_counter()
        return dets, {"pre_ms": (t1 - t0) * 1e3, "infer_ms": (t2 - t1) * 1e3, "post_ms": (t3 - t2) * 1e3}

    def warmup(self, n: int = 5):
        dummy = np.full((200, 200, 3), 128, np.uint8)
        for _ in range(n):
            self.infer(dummy)


def prepare_camera_frame(frame_bgr: np.ndarray, size: int = 200, to_gray: bool = True) -> np.ndarray:
    """Make a live-camera frame look like a NEU-DET crop: centre square crop -> size x size -> grey (3 channels).
    NEU-DET images are grey-looking 200x200 steel crops, so this reduces (but does not remove) the domain gap."""
    h, w = frame_bgr.shape[:2]
    s = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    crop = cv2.resize(frame_bgr[y0:y0 + s, x0:x0 + s], (size, size), interpolation=cv2.INTER_AREA)
    if to_gray:
        g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        crop = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    return crop

In [ ]:
%%writefile ../edge_inspect/pipeline.py
"""pipeline.py - image folder -> detector -> severity -> hand-off JSON -> Llama advice -> local log (+ tracker).

Everything runs locally/offline. The only network call is an OPTIONAL POST to the tracker on localhost.
Python 3.8+ compatible.
"""
from __future__ import annotations

import json
import time
import urllib.request
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import cv2

from .llama_advisor import advise
from .severity import build_handoff

SEV_BGR = {"Low": (107, 124, 62), "Medium": (27, 138, 201), "High": (46, 64, 179)}   # teal / amber / oxide red (BGR)


def annotate(img, handoff: dict):
    out = img.copy()
    for d in handoff["all_defects"]:
        x1, y1, x2, y2 = [int(v) for v in d["bbox_xyxy"]]
        col = SEV_BGR[d["severity"]]
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        cv2.putText(out, "%s %s %.2f" % (d["class"], d["severity"][0], d["confidence"]), (x1, max(10, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1, cv2.LINE_AA)
    return out


def post_json(url: str, payload: dict, timeout: float = 1.5) -> bool:
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=timeout).read()
        return True
    except Exception:
        return False        # tracker is optional; never break inspection because of it


def process_image(path: Path, detector, cfg: dict, backend=None, use_llama: bool = True,
                  conf_min: Optional[float] = None) -> dict:
    t_all = time.perf_counter()
    img = cv2.imread(str(path))
    if img is None:
        raise IOError("cannot read %s" % path)
    t_read = time.perf_counter()
    dets, tm = detector.infer(img)
    t_sev0 = time.perf_counter()
    h, w = img.shape[:2]
    handoff = build_handoff(Path(path).name, dets, w, h, cfg, conf_min=conf_min if conf_min is not None else detector.conf)
    t_sev1 = time.perf_counter()
    advice, llama_ms = None, 0.0
    if handoff["needs_advice"]:
        t0 = time.perf_counter()
        advice = advise(handoff, backend if use_llama else None)
        llama_ms = (time.perf_counter() - t0) * 1e3
    elif handoff["num_defects"] == 0:
        advice = advise(handoff, None)
    t_end = time.perf_counter()
    vision_ms = tm["pre_ms"] + tm["infer_ms"] + tm["post_ms"]
    return {
        "handoff": handoff, "advice": advice, "image": img,
        "timing_ms": {"read": round((t_read - t_all) * 1e3, 2), "pre": round(tm["pre_ms"], 2),
                      "infer": round(tm["infer_ms"], 2), "post": round(tm["post_ms"], 2),
                      "severity": round((t_sev1 - t_sev0) * 1e3, 3), "vision_total": round(vision_ms, 2),
                      "llama_total": round(llama_ms, 1), "end_to_end": round((t_end - t_all) * 1e3, 1)},
    }


def run_folder(folder: str, detector, cfg: dict, out_root: str = "runs", backend=None, use_llama: bool = True,
               tracker_url: Optional[str] = None, limit: Optional[int] = None, save_images: bool = True) -> dict:
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    files = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
    if limit:
        files = files[:limit]
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path(out_root) / run_id
    (out_dir / "annotated").mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "inspections.jsonl"
    records: List[dict] = []
    with open(log_path, "a") as log:
        for p in files:
            r = process_image(p, detector, cfg, backend, use_llama)
            ann_rel = None
            if save_images:
                ann_path = out_dir / "annotated" / (p.stem + "_ann.jpg")
                cv2.imwrite(str(ann_path), annotate(r["image"], r["handoff"]))
                ann_rel = str(ann_path.relative_to(out_root))
            rec = {"run_id": run_id, "handoff": r["handoff"], "advice": r["advice"],
                   "timing_ms": r["timing_ms"], "annotated_image": ann_rel, "source_image": str(p)}
            log.write(json.dumps(rec) + "\n")
            log.flush()
            if tracker_url:
                post_json(tracker_url.rstrip("/") + "/api/inspections", rec)
            records.append(rec)
    return {"run_id": run_id, "out_dir": str(out_dir), "log": str(log_path), "records": records}

In [ ]:
import urllib.request, cv2
import matplotlib.pyplot as plt
from edge_inspect import settings, severity as sv, detector as dt, llama_advisor as la, pipeline as pl
for m in (sv, dt, la, pl): importlib.reload(m)

S = settings.load(REPO)
meta = json.loads(Path(S["meta"]).read_text())
cfg = sv.load_config(S["severity_config"])

CONF = meta["deploy_conf"]            # confidence threshold chosen in notebook 00 (override here if you like, e.g. 0.25)
USE_LLAMA = True                      # False = detector + severity + template text only (fast)
LIMIT = 10                            # images to process; Llama is the slow part, raise this once everything works

det = dt.OnnxYoloDetector(S["onnx"], meta["classes"], imgsz=meta["imgsz"], conf=CONF, iou=meta["nms_iou"])
det.warmup(5)
print("ONNX Runtime providers in use:", det.providers, "| confidence threshold:", CONF)

backend = la.RunnerBackend(S["llama_main"], S["llama_model"], S["llama_tokenizer"], temperature=0.2)
if USE_LLAMA and not backend.available():
    print("Llama runner not found -> using template text. Finish notebook 02 to enable Llama.")
    backend = None

def tracker_up(url):
    try:
        urllib.request.urlopen(url.rstrip("/") + "/api/health", timeout=1).read(); return True
    except Exception:
        return False
TRACKER = S["tracker_url"] if tracker_up(S["tracker_url"]) else None
print("tracker:", TRACKER or "not running (the local log still works)")

In [ ]:
res = pl.run_folder(S["demo_images"], det, cfg, out_root=S["runs_dir"], backend=backend, use_llama=USE_LLAMA,
                    tracker_url=TRACKER, limit=LIMIT)
print("run:", res["run_id"], "| log:", res["log"])

recs = res["records"]
from collections import Counter
print("image severity:", dict(Counter(r["handoff"]["image_severity"] for r in recs)))
print("advice source :", dict(Counter((r["advice"] or {}).get("source", "none") for r in recs)))
for k in ["vision_total", "llama_total", "end_to_end"]:
    v = [r["timing_ms"][k] for r in recs]; print(f"{k:13s} mean {sum(v)/len(v):9.1f} ms   max {max(v):9.1f} ms")

In [ ]:
# Look at the results: annotated image + the log entry Llama wrote
n = min(8, len(recs)); cols = 4; rows_ = (n + cols - 1) // cols
fig, axs = plt.subplots(rows_, cols, figsize=(4 * cols, 4.4 * rows_)); axs = axs.ravel() if n > 1 else [axs]
for ax, r in zip(axs, recs[:n]):
    im = cv2.imread(str(Path(S["runs_dir"]) / r["annotated_image"]))
    ax.imshow(im[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"{r['handoff']['image_id']}\n{r['handoff']['image_severity']}", fontsize=9)
for ax in axs[n:]: ax.axis("off")
plt.tight_layout(); plt.show()
for r in recs[:5]:
    a = r["advice"] or {}
    print(f"[{a.get('source')}] {a.get('log_entry')}\n    why   : {a.get('explanation')}\n    do    : {a.get('actions')}\n")

## Optional: live-camera frames

NEU-DET images are 200x200 grey-looking steel crops. `prepare_camera_frame` takes a centre square of a camera frame, resizes it to 200x200 and converts it to grey, so the detector sees something closer to its training data.
Expect a **domain gap**: lighting, texture and scale differ from the dataset, so confidence and recall can drop. Judge it on a few real steel samples before the demo, and mention the gap in the report.

In [ ]:
RUN_CAMERA = False            # set True to grab one frame from camera 0
if RUN_CAMERA:
    cap = cv2.VideoCapture(0); ok, frame = cap.read(); cap.release()
    assert ok, "camera not available"
    crop = dt.prepare_camera_frame(frame, size=200, to_gray=True)
    out = Path(S["runs_dir"]) / "camera_frame.jpg"; out.parent.mkdir(exist_ok=True); cv2.imwrite(str(out), crop)
    r = pl.process_image(out, det, cfg, backend, use_llama=USE_LLAMA)
    print(json.dumps({k: v for k, v in r["handoff"].items() if k != "all_defects"}, indent=2)); print(r["advice"])
    plt.imshow(pl.annotate(r["image"], r["handoff"])[:, :, ::-1]); plt.axis("off"); plt.show()